# Estrategia y Flujo de Trabajo en Competencias Kaggle

## 🎯 Objetivos de Aprendizaje
- Comprender la anatomía de una competencia en la plataforma Kaggle.
- Diferenciar entre los conjuntos de datos de entrenamiento (`train`), prueba (`test`) y el formato de sumisión (`submission.csv`).
- Establecer una validación cruzada local robusta para evitar sobreajuste al *Public Leaderboard*.
- Construir un pipeline base (*baseline model*) reproducible con Scikit-Learn y Pandas.
- Generar y verificar un archivo de predicciones conforme a los requerimientos de la competencia.

## 🌉 Puente Pedagógico: Del Análisis Exploratorio al Entorno Competitivo

### ¿Por qué participar en Kaggle?
Kaggle ofrece problemas reales planteados por empresas de clase mundial con métricas de evaluación objetivas. Permite poner a prueba habilidades de ingeniería de variables, modelado y validación cruzada frente a la comunidad internacional.

### Analogía
Una competencia Kaggle es como una carrera de atletismo de relevos:
- **Fase 1 (EDA)**: Reconocer la pista y el terreno.
- **Fase 2 (Feature Engineering)**: Diseñar el calzado y preparación óptima.
- **Fase 3 (Modelo & Ensembling)**: El corredor que entrega la máxima velocidad posible.
- **Fase 4 (Submission)**: Cruzar la meta cumpliendo con el reglamento estricto sin descalificación.

### Diagrama del Flujo Estándar de Competencia
```
  +-------------------------------------------------------+
  |                    DATASET KAGGLE                     |
  |  +-------------------------+  +--------------------+  |
  |  | Train (con variable Y)  |  | Test (sin Y)       |  |
  +--+------------+------------+--+---------+----------+--+
                  |                         |
                  v                         v
         [ Preprocesamiento ]     [ Mismo Preprocesamiento ]
                  |                         |
                  v                         |
        [ Cross-Validation K-Fold ]         |
                  |                         |
                  v                         |
           [ Modelo Final ] <---------------+ (Inferencia)
                  |                         |
                  +-------------------------+
                               |
                               v
                  [ submission.csv (id, y_pred) ]
                               |
                               v
                  [ Subir a Kaggle / Leaderboard ]
```

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Simulación de pipeline Kaggle: Predicción de Demanda
np.random.seed(42)
n_train = 500
n_test = 100

# Dataset de entrenamiento con etiqueta target 'demanda'
df_train = pd.DataFrame({
    "id": range(1, n_train + 1),
    "precio": np.random.uniform(10, 50, n_train),
    "promocion": np.random.choice([0, 1], p=[0.7, 0.3], size=n_train),
    "dia_semana": np.random.choice(range(1, 8), size=n_train)
})
df_train["demanda"] = 200 - 3 * df_train["precio"] + 40 * df_train["promocion"] + np.random.normal(0, 5, n_train)

# Dataset de prueba sin target
df_test = pd.DataFrame({
    "id": range(n_train + 1, n_train + n_test + 1),
    "precio": np.random.uniform(10, 50, n_test),
    "promocion": np.random.choice([0, 1], p=[0.7, 0.3], size=n_test),
    "dia_semana": np.random.choice(range(1, 8), size=n_test)
})

print("--- Vista previa del dataset de entrenamiento ---")
display(df_train.head())

## 1. Validación Local y Entrenamiento del Baseline

In [ ]:
features = ["precio", "promocion", "dia_semana"]
X = df_train[features]
y = df_train["demanda"]

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = RandomForestRegressor(n_estimators=100, random_state=42)
modelo.fit(X_tr, y_tr)

preds_val = modelo.predict(X_val)
rmse_val = np.sqrt(mean_squared_error(y_val, preds_val))
print(f"Error RMSE de Validación Local: {rmse_val:.4f}")

## 2. Generación del Archivo de Envío (`submission.csv`)

El archivo debe respetar exactamente los nombres de columnas y el total de filas que exige la competencia.

In [ ]:
# Predicción en el test set ciego
preds_test = modelo.predict(df_test[features])

submission = pd.DataFrame({
    "id": df_test["id"],
    "demanda": np.round(preds_test, 2)
})

submission.to_csv("submission.csv", index=False)
print(f"Archivo submission.csv generado con éxito ({len(submission)} filas):")
display(submission.head())

## 📋 Resumen y Mejores Prácticas en Competencias
- **Confía en tu CV (Cross-Validation)**: Nunca tomes decisiones basadas únicamente en el Leaderboard público, pues suele inducir a overfitting severo.
- **Feature Engineering supera a algoritmos complejos**: La creación de variables de interacción, agrupaciones y agregaciones suele aportar más mejoras que ajustar hiperparámetros finos.
- **Control de integridad**: Comprueba siempre que tu archivo de sumisión no tenga valores nulos (`isna().sum() == 0`) antes de cargarlo.